# Special Topic — Deep Learning (Awareness, When-to-Use, and One Tabular Demo)

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/mgmt_474_ai_logo_02-modified.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>QM47400 Predictive Analytics</center>
# <center>Professor: Davi Moreira </center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/blob/main/notebooks/nb19_deep_learning_student.ipynb)


## Learning Objectives

By the end of this notebook, you will be able to:

1. Explain the historical arc that took neural networks from the 1980s rebrand into deep learning's 2010 resurgence and the three drivers that made it work (compute, data, frameworks).
2. Describe what a single neuron, a single hidden layer, and a multi-layer perceptron (MLP) compute, in plain language.
3. Distinguish three deep-learning structural inventions — fully-connected MLPs, convolutional networks (CNNs), and recurrent networks (RNNs) — and name one problem class each is designed for.
4. Decide whether deep learning is the right tool for a given business problem using a four-question rubric.
5. Run a single MLP classifier on a familiar tabular dataset and compare it honestly to gradient boosting from nb13.


> **📋 Participation Reminder:** This notebook contains **2 PAUSE-AND-DO exercises**. Complete both to receive participation credit.

> 📝 *This lecture's content is inspired by and replicates material from [An Introduction to Statistical Learning (ISLP)](https://www.statlearning.com/), the textbook the rest of the course leans on.*


## 💼 Why This Matters: The "What About AI?" Question Every Analyst Will Hear

The **VP of Strategy at TechCorp** sat in your Milestone 4 poster session, watched the gradient-boosting churn model demo, and asked one question:

> *"This is great. But shouldn't we be using deep learning?"*

This notebook is the answer you owe her. It is not a course in PyTorch — that is a semester on its own. It is a working analyst's awareness module: enough of the language and the structural ideas that you can recognize when deep learning **is** the right tool, recognize when it **is not** (most tabular business problems), and run a single fair comparison so the answer to her question is evidence-based, not vibes.

The three structural inventions the field rallies around — **fully-connected MLPs**, **convolutional networks (CNNs) for images**, and **recurrent networks (RNNs) for sequences** — get a one-section explanation each, with the figures from the course's deep-learning slide deck. We close with a four-question rubric for "is deep learning right for this problem?" and one MLP demo on familiar tabular data so you can speak from experience, not lecture notes.

<center>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/dl_pioneers.png" alt="Deep learning pioneers" width="500"/>
<br>
<small>The 2019 ACM Turing Award honored Yann LeCun, Geoffrey Hinton, and Yoshua Bengio for the work that became modern deep learning.</small>
</center>


## 1. The Historical Arc — From 1980s Rebrand to 2010 Resurgence

Neural networks have lived two lives. **Round one (1980s):** a community of researchers led by LeCun, Hinton, and Bengio invented backpropagation, organized around conferences like NeurIPS, and showed early successes on small problems. Then in the 1990s, simpler methods (SVMs, Random Forests, Boosting) outperformed neural nets on most benchmarks, and the field receded.

**Round two (2010 onward):** three things changed at once.

1. **Compute** — GPUs designed for video games turned out to be the right hardware for matrix multiplications, and a $1,000 graphics card replaced a $100,000 cluster.
2. **Data** — ImageNet, with 14 million labeled images, gave neural nets enough training signal to outperform every classical method.
3. **Frameworks** — TensorFlow (Google, 2015) and PyTorch (Meta, 2016) reduced the boilerplate from "thousands of lines of CUDA" to "a few lines of Python."

Combine those three and the neural networks of the 1980s, mostly unchanged, suddenly worked. The rebrand to **deep learning** is mostly marketing; the math is older than most of you. What changed is the engineering stack around the math.

> **A question that often comes up here:** *"If the math is unchanged, why did it take 25 years?"* Because the math was *almost* unchanged. Three small but load-bearing additions — ReLU activations replacing sigmoid (which trained better in deep networks), dropout regularization, batch normalization — combined with the compute and data to make training networks with hundreds of layers practical for the first time. Each addition is a "small" idea, but together they crossed the threshold.


## 2. PyTorch and TensorFlow — Two Frameworks, One Idea

Today, almost every deep-learning model in production sits on top of either **PyTorch** (developed at Meta, dominant in research and increasingly in production) or **TensorFlow** (developed at Google, historically dominant in production). They differ in style and ecosystem but not in capability.

<center>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/dl_pytorch_vs_tensorflow.png" alt="PyTorch vs. TensorFlow" width="600"/>
</center>

**What both give you (and why neither is `numpy`):**

- **Tensor operations.** A tensor is a multi-dimensional array (`numpy.ndarray` is a 1- or 2-D version; tensor frameworks scale to 4-D image batches and 5-D video batches without breaking a sweat).
- **Automatic differentiation.** When you write `loss = something(weights)`, the framework remembers the chain of operations and computes `dloss/dweights` for you with one call (`loss.backward()` in PyTorch). That is the magic that makes training a network with millions of parameters tractable.
- **GPU support.** Move a tensor to GPU with `.to('cuda')` (PyTorch) and every subsequent operation runs on the GPU. CPU code "just" runs \~50× slower.
- **Pre-built layers.** Convolution, recurrence, attention — none of those need to be implemented from scratch. The framework supplies a library of named building blocks.

**For a working business analyst:** if you have to pick one, pick **PyTorch**. Research code on GitHub, the Hugging Face model hub, and the modern LLM ecosystem are all PyTorch-first. TensorFlow is still common in production at Google-scale companies, but the gap is closing every year.


## 3. Single Neuron → Single Layer → Multi-Layer Perceptron (MLP)

A **neuron** computes a weighted sum of its inputs, adds a bias, and passes the result through a non-linear "activation" function. Three numbers in (or three thousand), one number out. That is the entire content of a neuron.

<center>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/dl_neuron.png" alt="Single neuron" width="450"/>
</center>

A **single hidden layer** is a row of neurons that share the same input but have different weights. With enough neurons in one hidden layer, the network can in theory approximate any continuous function ("universal approximation theorem"). In practice, "enough" is large and slow to train, which is why we stack multiple layers.

<center>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/dl_single_layer.png" alt="Single hidden layer" width="500"/>
</center>

A **multi-layer perceptron (MLP)** stacks several hidden layers. Each layer transforms its input into a more abstract representation; the final layer makes the prediction. "Deep learning" is mostly a marketing term for "MLP with many hidden layers" plus the structural innovations in the next two sections.

<center>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/dl_multi_layer.png" alt="Multi-layer network" width="600"/>
</center>

> **A question that often comes up here:** *"How is an MLP different from a logistic regression?"* A logistic regression is structurally an MLP with **zero hidden layers** and a sigmoid output. Add one hidden layer with a non-linear activation and the model can learn interactions between features automatically — no more manually engineering `feature_a * feature_b` columns. That is the headline value an MLP adds for tabular data.


## 4. Convolutional Networks (CNNs) — When the Data Is an Image

A CNN is an MLP with one structural change: instead of every neuron in a layer connecting to every neuron in the previous layer, neurons connect only to a **local window** of the previous layer's output. The window slides across the image (the **convolution** operation), and the same weights are reused at every position. Two consequences:

1. **Translation invariance.** A cat in the top-left corner activates the same neurons as a cat in the bottom-right corner. The model does not need to relearn "what a cat looks like" at every position.
2. **Parameter efficiency.** A fully-connected layer connecting a 1000×1000 image to 1000 hidden units would have 1 billion weights. A convolutional layer with a 3×3 window and 1000 hidden filters has 9000 weights. CNNs make image learning tractable.

<center>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/dl_cnn_overview.png" alt="CNN overview" width="600"/>
</center>

CNNs power image classification (CIFAR, ImageNet), object detection (autonomous vehicles, retail shelf monitoring), and medical imaging (radiology, pathology). The CIFAR-100 benchmark below is the kind of task CNNs were built for — a fully-connected MLP would also work, but at hundreds of times the compute cost and worse accuracy.

<center>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/dl_cifar100.png" alt="CIFAR-100" width="500"/>
</center>


## 5. Recurrent Networks (RNNs) — When the Data Is a Sequence

An RNN is an MLP that processes inputs **one at a time** and carries forward a "hidden state" that summarizes everything seen so far. At step *t*, the network sees input *t* and the hidden state from step *t−1*; it produces output *t* and an updated hidden state for step *t+1*.

<center>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/dl_rnn_overview.png" alt="RNN overview" width="600"/>
</center>

This structural change makes RNNs the natural fit for **sequence data**: text (words arrive one at a time), audio (samples arrive thousands of times per second), and time series (months arrive in order, exactly like nb16's DemandCo data).

**Three variants you will hear named:**

- **Vanilla RNN** — the original; trains poorly on long sequences because gradients vanish.
- **LSTM** (Long Short-Term Memory) — adds a gating mechanism that lets the network learn what to remember and what to forget. The default for sequence problems from 2014 to \~2018.
- **Transformer** — replaces recurrence with attention, processes the whole sequence in parallel, and now dominates language and increasingly time-series. ChatGPT, Claude, and every modern LLM are transformers.

> **A question that often comes up here:** *"Should I use an RNN for the DemandCo time series in nb16?"* For 60 monthly observations, no. RNNs need hundreds-to-thousands of points to outperform a well-engineered linear lag-feature model, and even then the gain is often within the CV confidence interval. Use deep learning for sequences when you have *tens of thousands* of time steps (high-frequency trading, sensor data) or when the dependence is highly non-linear.


## 📝 PAUSE-AND-DO Exercise 1 — When Is Deep Learning the Right Tool? (10 minutes)

**Task:** Apply the four-question rubric below to **two** problems: the Bank Churn case competition (nb18) and the DemandCo monthly forecast (nb16). For each problem, answer the four questions and produce one verdict — *deep learning is/is not the right primary tool*.

**The four-question rubric:**

1. **Data shape.** Is the input an image, audio, long text sequence, or a high-dimensional structured object (e.g., a graph)? If yes, lean DL. If it is a tabular row, lean classical ML.
2. **Sample size.** Do you have at least tens of thousands of labeled examples (and ideally millions)? If yes, DL has the data it needs to outperform. If you have hundreds or low thousands, classical ML almost always wins.
3. **Compute budget.** Can you spend \~10× the training time and \~50× the inference cost vs. a tree ensemble? If yes, DL is in scope. If you need a model that fits on a CPU and predicts in milliseconds, prefer classical ML.
4. **Interpretability requirement.** Does the stakeholder need a feature-by-feature explanation for every prediction? If yes, DL adds friction (post-hoc explainers like SHAP exist but are imperfect). If predictions only need to be accurate, DL is in scope.

**Verdict rule:** Answer "yes" to **at least three** questions for DL to be the right primary tool. Otherwise, classical ML first.


### YOUR RUBRIC ANSWERS HERE:

**Problem A — Bank Churn case competition (tabular, \~10K rows, churn probability for retention team):**

1. Data shape: *[tabular / sequential / image — and DL lean?]*
2. Sample size: *[count + lean]*
3. Compute budget: *[lean]*
4. Interpretability: *[lean]*
5. **Verdict:** *[DL primary / Classical ML primary]*  *[one sentence why]*

**Problem B — DemandCo monthly demand forecast (60 months, procurement target):**

1. Data shape: *[lean]*
2. Sample size: *[lean]*
3. Compute budget: *[lean]*
4. Interpretability: *[lean]*
5. **Verdict:** *[DL primary / Classical ML primary]*  *[one sentence why]*


## 6. One Honest Demo — `MLPClassifier` vs. Gradient Boosting on Tabular Data

We close with the comparison the VP of Strategy actually wants: a feed-forward neural network on the same Bank-Churn-style classification problem, evaluated on the same CV folds, against gradient boosting from nb13. We use scikit-learn's `MLPClassifier` (an off-the-shelf MLP) so you can run this on a CPU in under a minute — no PyTorch installation required.


In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from scipy import stats
from sklearn.datasets import make_classification
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import GradientBoostingClassifier

RANDOM_SEED = 474
np.random.seed(RANDOM_SEED)

# Synthetic tabular classification — same shape as nb15
X, y = make_classification(
    n_samples=5000, n_features=20, n_informative=15, n_redundant=5,
    n_classes=2, random_state=RANDOM_SEED,
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

mlp_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        max_iter=200,
        random_state=RANDOM_SEED,
    )),
])
gbm = GradientBoostingClassifier(
    n_estimators=300, max_depth=3, learning_rate=0.05, random_state=RANDOM_SEED,
)

mlp_aucs = cross_val_score(mlp_pipe, X, y, scoring="roc_auc", cv=cv, n_jobs=-1)
gbm_aucs = cross_val_score(gbm, X, y, scoring="roc_auc", cv=cv, n_jobs=-1)

k = 5
t_crit = stats.t.ppf(0.975, df=k - 1)  # ≈ 2.776 — same Student's t constant from nb08

def _row(scores):
    m = float(scores.mean()); sd = float(scores.std(ddof=1))
    half_w = t_crit * sd / np.sqrt(k)
    return {'AUC_mean': m, 'AUC_sd': sd, 'AUC_half_w': half_w,
            'AUC_ci_low': m - half_w, 'AUC_ci_high': m + half_w}

summary = pd.DataFrame({
    'MLPClassifier (64,32)': _row(mlp_aucs),
    'GradientBoosting':      _row(gbm_aucs),
}).T
print(summary)

# CI-overlap verdict
mlp = summary.loc['MLPClassifier (64,32)']
gbm_row = summary.loc['GradientBoosting']
overlap = not (mlp['AUC_ci_high'] < gbm_row['AUC_ci_low'] or gbm_row['AUC_ci_high'] < mlp['AUC_ci_low'])
if not overlap:
    winner = 'MLPClassifier' if mlp['AUC_mean'] > gbm_row['AUC_mean'] else 'GradientBoosting'
    print(f'\nVerdict: CIs do NOT overlap → {winner} wins outright (CI-clear margin).')
else:
    print('\nVerdict: CIs overlap → statistical tie → simpler / faster model wins by default.')


**Reading the output:**

On a 5,000-row tabular problem with 20 features, the MLP and the gradient-boosted ensemble usually land within a few thousandths of each other — well inside the CV 95% CI. That overlapping interval is the empirical answer to the VP's question: deep learning **does not** outperform a tuned tree ensemble on this kind of data, and the ensemble is faster, more interpretable, and easier to deploy.

The story flips when the input is an image, a long text passage, or a high-frequency sensor stream. There the MLP's flexibility (or, more typically, a CNN's or transformer's structural priors) starts paying for itself. The rubric in section 5 captures the practical decision rule.

> **A question that often comes up here:** *"What if I give the MLP more layers?"* You can — try `hidden_layer_sizes=(128, 64, 32)` or `(256, 128, 64, 32)`. On this data the AUC will not move outside the CI; what will move is training time and the chance of training instability. Bigger is not better when the data structure does not need bigger.


## 📝 PAUSE-AND-DO Exercise 2 — Compare and Decide (10 minutes)

**Task:** Add a third candidate to the comparison table — a deeper MLP with `hidden_layer_sizes=(128, 64, 32)` — and rerun the CV. Then write three sentences answering the VP of Strategy's question.

**Hints:**
- Reuse `mlp_pipe` style: only the `hidden_layer_sizes` argument changes.
- Do not change the CV splits — identical folds are what makes the comparison honest.


In [ ]:
# YOUR SOLUTION CODE HERE

# Hints:
# deep_pipe = Pipeline([
#     ("scaler", StandardScaler()),
#     ("clf", MLPClassifier(
#         hidden_layer_sizes=(128, 64, 32), activation="relu",
#         max_iter=300, random_state=RANDOM_SEED,
#     )),
# ])
# deep_aucs = cross_val_score(deep_pipe, X, y, scoring="roc_auc", cv=cv, n_jobs=-1)
# Build the 3-row summary table and inspect CIs.


### YOUR ANSWER TO THE VP HERE:

**Three sentences for the VP of Strategy:**

1. *[Empirical comparison — which model won on this data and by how much in CV CI terms?]*
2. *[Why this generalizes to the Bank Churn problem — data shape and sample size argument from the rubric]*
3. *[Where deep learning **would** earn its keep at TechCorp — name one concrete use case from the company's likely data assets]*


## 7. Wrap-Up — Key Takeaways

1. **Deep learning is the *engineering stack* around old neural-network math.** Compute + data + frameworks were the bottleneck; the math was mostly already there.
2. **The three structural inventions are MLP, CNN, and RNN/Transformer.** MLP for tabular extras, CNN for images, RNN/Transformer for sequences. If you can name the right one for a problem, you have \~80% of what an analyst needs in this conversation.
3. **For tabular business problems with thousands of rows, gradient boosting almost always wins.** The MLP is a credible candidate but rarely the champion. The four-question rubric in section 5 keeps you honest.
4. **PyTorch is the safer default to learn next.** The Hugging Face model hub, almost every research paper, and the modern LLM ecosystem are PyTorch-first.

> **A question that often comes up here:** *"What about ChatGPT, Claude, and the LLMs we use every day in this course?"* They are transformers (a successor to RNNs) trained on enormous text corpora. As an analyst, you will mostly **use** them through APIs (you have done this all course via Gemini prompts) rather than train them. The "knowing the structural ideas" bar from this notebook is the right bar — it lets you read the docs and the announcement papers without getting lost.

**Next stop — nb20: Course End and Reflection.** Tomorrow is delivery day: M4 poster, Kaggle final submission, intra-group peer evaluation, and the reflection survey that closes the course. Today's awareness gives you the language for the "what's next?" line on your poster, and one credible answer when a colleague asks "should we be doing deep learning?".


## Participation Assignment Submission Instructions

1. **Complete both PAUSE-AND-DO exercises**.
2. **Run all cells**.
3. **Save with output** and submit `nb19_deep_learning_<your_lastname>.ipynb` to Brightspace.

**Bibliography**
- ISLP, Chapter 10: Deep Learning (the textbook chapter behind this notebook's lecture slides).
- PyTorch tutorials: <https://pytorch.org/tutorials/beginner/basics/intro.html>.
- *Deep Learning* by Goodfellow, Bengio, and Courville (free online): <https://www.deeplearningbook.org/>.
- 3Blue1Brown's neural-network video series (the best visual intuition online).


<center>

# Thank you!

</center>
